In [1]:
import pandas as pd
import ast
import os
from Bio import SeqIO

def extract_genus_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_genus_name)

genus_name = 'Escherichia'

org_data_n = all_data[all_data['genus_clean'].str.contains(genus_name, na=False)].reset_index(drop=True)
pla_acc = []
for i in org_data_n.index:
    acc_n = org_data_n['accession'][i]
    pla_data = ast.literal_eval(org_data_n['plasmid contigs'][i])
    for item in pla_data:
        pla_acc.append(acc_n + '-' + item)
        

acc_bio = pd.read_csv('/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/biosample_info.tsv', sep='\t')
acc_time = pd.read_csv('/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/assembly_submission_info.tsv', sep='\t')
acc_time["submissionDate"] = pd.to_datetime(acc_time["submissionDate"])

target_dir = f'/active-data/analysis_results/chr_pla/genus/statistics_records/{genus_name}'
replicon_data = pd.read_csv(f'{target_dir}/replicon-plasmid_fraction-self_bitscore_statistics.csv')
all_trans = replicon_data[replicon_data['category-pident_90']=='intermediate replicon'].copy()
all_trans['acc_n'] = all_trans['accession'].str.split('-').str[0]
all_acc = set(all_trans['acc_n'])

filted_bio = acc_bio[(acc_bio['accession'].isin(all_acc)) & (acc_bio['srr_list'] != '[]')]
filted_bio = pd.merge(filted_bio, acc_time, how='left', on='accession')
filted_bio = filted_bio.sort_values(by="submissionDate", ascending=False, ignore_index=True)

all_acc = []
for idx in filted_bio.index:
    short_read, long_read = False, False
    srr_list = ast.literal_eval(filted_bio.loc[idx, 'srr_list'])
    for run in srr_list:
        if run['platform'] == 'ILLUMINA':
            short_read = True
        if run['platform'] == 'OXFORD_NANOPORE' or run['platform'] == 'PACBIO_SMRT':
            long_read = True
    if short_read and long_read:
        pass
    else:
        continue

    all_acc.append(filted_bio.loc[idx, 'accession'])

In [2]:
def seq_frac_calcu(align_result, lenth, pla_acc):
    temp_id_list = list(align_result['pident'].value_counts().index)
    add_counts_a = align_result['qstart'].value_counts()
    minor_counts_a = align_result['qend'].value_counts()
    
    align_result = align_result[align_result['sseqid'].isin(pla_acc)]
    add_counts_p = align_result['qstart'].value_counts()
    minor_counts_p = align_result['qend'].value_counts()
    
    add_num_a = 0
    add_num_p = 0
    fra_list = []
    for j in range(lenth):
        if j+1 in add_counts_a.index:
            add_num_a += add_counts_a[j+1]
        all_count = int(add_num_a)
        if j+1 in minor_counts_a.index:
            add_num_a -= minor_counts_a[j+1]
            
        if j+1 in add_counts_p.index:
            add_num_p += add_counts_p[j+1]
        pla_count = int(add_num_p)
        if j+1 in minor_counts_p.index:
            add_num_p -= minor_counts_p[j+1]
            
        try:
            fra = pla_count/all_count
        except:
            fra = 1
        fra_list.append(fra)
    return fra_list

def pla_frac_calcu(acc_n, genus_name, pla_acc, que):
    folder = f'/active-data/analysis_results/chr_pla/genus/re-assemble_fraction_records/{genus_name}/{acc_n}'
    unicyle_dir = f'/active-data/genome_re-assemble/{acc_n}/assembly_unicycler/assembly.fasta'
    hybracter_dir = f'/active-data/genome_re-assemble/{acc_n}/assembly_hybracter/FINAL_OUTPUT/complete/{acc_n}_final.fasta'
    hybracter_im_dir = f'/active-data/genome_re-assemble/{acc_n}/assembly_hybracter/FINAL_OUTPUT/incomplete/{acc_n}_final.fasta'
    flye_dir = f'/active-data/genome_re-assemble/{acc_n}/assembly_flye/assembly.fasta'
    for file_dir, target_fold in zip([unicyle_dir, hybracter_dir, hybracter_im_dir, flye_dir], ['unicyle', 'hybracter', 'hybracter', 'flye']):
        try:
            handle = open(file_dir)
            os.makedirs(f'{folder}/{target_fold}', exist_ok=True)
        except:
            continue

        csv_path = f"{folder}/{target_fold}/contig_average_plasmid_fraction.csv"
        if os.path.isfile(csv_path):
            continue
            
        records = SeqIO.parse(handle, "fasta")
        all_result = []
        for seq_record in records:
            seq_record.id = f'{acc_n}_{seq_record.id}'
            
            frac_list = {'original':[], 'pident_90':[], 'pident_95':[]}
            average_value = {'accession': acc_n, 'contig': seq_record.id, 'size': len(seq_record)}
            os.chdir('/active-data/temp/blastn')
            temp_ncl_file = open(f'temp_nucleotide_seq_{acc_n}.fasta', 'w+')
            SeqIO.write(seq_record, temp_ncl_file, "fasta")
            temp_ncl_file.close()
            os.system(f'blastn -query temp_nucleotide_seq_{acc_n}.fasta -db /active-data/analysis_results/chr_pla/genus/blast_data/{genus_name}/nucleotide_seq.blastdb -out blastn_results_{acc_n}.txt -evalue 1e-50 -max_target_seqs 100000 -max_hsps 3000 -outfmt 6 -num_threads 8')
            head = ['qseqid', 'sseqid', 'pident', 'length', 'mismatch', 'gapopen', 'qstart', 'qend', 'sstart', 'send', 'evalue', 'bitscore']
            align_result = pd.read_csv(f'blastn_results_{acc_n}.txt', sep = '\t|;', engine = 'python', header = None, names = head)

            for dir_label in frac_list:
                if '90' in dir_label:
                    fra_list = seq_frac_calcu(align_result[align_result['pident'] >= 90], len(seq_record), pla_acc)
                elif '95' in dir_label:
                    fra_list = seq_frac_calcu(align_result[align_result['pident'] >= 95], len(seq_record), pla_acc)
                else:
                    fra_list = seq_frac_calcu(align_result, len(seq_record), pla_acc)
                tot_count = sum(fra_list)
                average_value[f'average plasmid fraction-{dir_label}'] = tot_count/len(seq_record)
            all_result.append(pd.DataFrame([average_value]))
        handle.close()
        all_value = pd.concat(all_result, ignore_index=True)
        os.chdir(f'{folder}/{target_fold}')
        all_value.to_csv('contig_average_plasmid_fraction.csv', index=False)
    que.put(1)

In [3]:
from tqdm import tqdm
import multiprocessing

manager = multiprocessing.Manager()
que = manager.Queue()

par = 10
tot = len(all_acc)
pool = multiprocessing.Pool(par)

for acc_n in all_acc:
    pool.apply_async(pla_frac_calcu, (acc_n, genus_name, pla_acc, que))
    
pool.close()

count = 0
with tqdm(total = tot, desc=f'{genus_name}({tot})', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
    while True:
        if not que.empty():
            value = que.get(True)
            count += 1
            pbar.update(1)
            if count == tot:
                break
        else:
            continue

pool.join()

Escherichia(36): 100%|███████████████████████████████████████████| 36.0/36.0 [2:49:33<00:00, 283s/B]
